# Logit Lens

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_Mech_Interp/blob/main/exercises/01_logit_lens/notebook_normal.ipynb)

In [ ]:
import os, importlib

if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_Mech_Interp/main/exercises/01_logit_lens/utils.py",
        "utils.py"
    )
    %pip install "transformer_lens==3.9.0" einops matplotlib -q

import utils
importlib.reload(utils)
from utils import test_unembed, test_get_resid_stack, test_logit_lens, test_kl_from_final

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)

## The idea

GPT-2 builds its prediction in the **residual stream**: one vector per token position that every layer reads from and adds to.
Only at the very end is this vector turned into a prediction for the next token:

$$\text{logits} = W_U \, \mathrm{LN}_{\text{final}}(h_{\text{final}}) + b_U$$

The **logit lens** asks: *what if we stopped early?*
Take the residual stream $h_\ell$ after layer $\ell$ and push it through that same final step, skipping all remaining layers.
This works surprisingly well, because every layer only *adds* to the residual stream, so intermediate and final residual streams share the same coordinates.

You will build the lens in four small steps:
1. turn any residual stream vector into logits,
2. collect the residual stream at every layer,
3. apply step 1 at every layer,
4. measure how far each layer is from the model's final prediction.

In [ ]:
model = HookedTransformer.from_pretrained("gpt2-small")

## Looking inside: `run_with_cache`

`model.run_with_cache(tokens)` runs the model and also returns a `cache` with every intermediate activation.
The residual stream after block $\ell$ is `cache[f"blocks.{l}.hook_resid_post"]`, and the residual stream *before* block 0 (just the embeddings) is `cache["blocks.0.hook_resid_pre"]`.

In [ ]:
tokens = model.to_tokens("The quick brown fox jumps over the lazy")
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)

print("tokens:", tokens.shape, " logits:", logits.shape)
print("embeddings:      ", cache["blocks.0.hook_resid_pre"].shape)
print("after block 5:   ", cache["blocks.5.hook_resid_post"].shape)
print("after block 11:  ", cache["blocks.11.hook_resid_post"].shape, "<- the final residual stream")

## Exercise 1: From residual stream to logits

Write `unembed`, which applies the model's final step to a residual stream vector:
first the final LayerNorm `model.ln_final`, then the unembedding `model.unembed` (which computes $W_U x + b_U$).

It should work for any leading shape: a single `(d_model,)` vector, or `(seq_len, d_model)`, or more.

**Check:** applied to the final residual stream, it must reproduce the model's logits exactly.

In [ ]:
def unembed(model, resid):
    """(..., d_model) residual stream -> (..., d_vocab) logits"""
    # TODO: apply model.ln_final, then model.unembed
    return ...

In [ ]:
test_unembed(unembed, model)

<details>
<summary><b>Hint</b></summary>

Both `model.ln_final` and `model.unembed` are modules you can call like functions. Don't skip the LayerNorm: without it the top token often looks right, but every probability is wrong.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def unembed(model, resid):
    return model.unembed(model.ln_final(resid))
```
</details>

## Exercise 2: The residual stream at every layer

Collect the residual stream at every depth into one tensor of shape `(n_layers + 1, seq_len, d_model)`:
- index `0`: the embeddings, before any layer (`blocks.0.hook_resid_pre`),
- index `l + 1`: after block `l` (`blocks.{l}.hook_resid_post`).

The last entry is the final residual stream, the one the model actually unembeds.

In [ ]:
def get_resid_stack(model, tokens):
    """(1, seq_len) tokens -> (n_layers + 1, seq_len, d_model) residual streams"""
    _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    # TODO: make a list [embeddings, after block 0, ..., after block n_layers - 1]
    resids = ...
    return torch.stack(resids)

In [ ]:
test_get_resid_stack(get_resid_stack, model)

<details>
<summary><b>Hint</b></summary>

Start the list with `cache["blocks.0.hook_resid_pre"]`, then add `cache[f"blocks.{l}.hook_resid_post"]` for `l in range(model.cfg.n_layers)`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def get_resid_stack(model, tokens):
    _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    resids = [cache["blocks.0.hook_resid_pre"]] + [cache[f"blocks.{l}.hook_resid_post"] for l in range(model.cfg.n_layers)]
    return torch.stack(resids)
```
</details>

## Exercise 3: The logit lens

Now combine the two: apply `unembed` to the residual stream at every layer and return **log-probabilities** over the vocabulary, shape `(n_layers + 1, seq_len, d_vocab)`.

In [ ]:
def logit_lens(model, tokens):
    """(1, seq_len) tokens -> (n_layers + 1, seq_len, d_vocab) log-probabilities"""
    resids = get_resid_stack(model, tokens)
    # TODO: unembed every layer at once, then log_softmax over the vocabulary
    return ...

In [ ]:
test_logit_lens(logit_lens, model)

<details>
<summary><b>Hint</b></summary>

`unembed` already handles the extra leading dimension, so it is one line: `unembed(model, resids).log_softmax(dim=-1)`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def logit_lens(model, tokens):
    resids = get_resid_stack(model, tokens)
    return unembed(model, resids).log_softmax(dim=-1)
```
</details>

## Exercise 4: How far is each layer from the final answer?

To measure how close a layer's guess is to the model's actual output, we use the KL divergence *from the final distribution*:

$$D_{\mathrm{KL}}(P_{\text{final}} \,\|\, P_\ell) = \sum_v P_{\text{final}}(v)\,\big(\log P_{\text{final}}(v) - \log P_\ell(v)\big)$$

Write it directly from the formula.
Input: log-probabilities `(n_layers + 1, seq_len, d_vocab)`, as returned by `logit_lens`.
Output: `(n_layers + 1, seq_len)`.
The last row compares the final layer with itself, so it must be 0.

In [ ]:
def kl_from_final(log_probs):
    """(n_layers + 1, seq_len, d_vocab) log-probs -> (n_layers + 1, seq_len) KL from the final layer"""
    final = log_probs[-1]  # (seq_len, d_vocab), broadcasts against every layer
    # TODO: implement the formula, summing over the vocabulary
    return ...

In [ ]:
test_kl_from_final(kl_from_final)

<details>
<summary><b>Hint</b></summary>

$P_{\text{final}}$ is `final.exp()`. Multiply it with `final - log_probs` and `.sum(dim=-1)`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def kl_from_final(log_probs):
    final = log_probs[-1]
    return (final.exp() * (final - log_probs)).sum(dim=-1)
```
</details>

## Visualizing the logit lens

Each cell shows the top prediction of one layer at one position.
The colour is the KL from the final prediction: white means the layer already agrees with the model's output.

In [ ]:
def plot_logit_lens(model, text):
    tokens = model.to_tokens(text)
    log_probs = logit_lens(model, tokens)
    kl = kl_from_final(log_probs).cpu()
    top = log_probs.argmax(-1).cpu()
    n_rows, seq_len = top.shape
    row_names = ["embed"] + [f"after L{l}" for l in range(n_rows - 1)]

    fig, ax = plt.subplots(figsize=(max(8, 1.1 * seq_len), 0.45 * n_rows + 1))
    im = ax.imshow(kl, cmap="Reds", aspect="auto", origin="lower", vmax=kl[1:].max())
    for l in range(n_rows):
        for t in range(seq_len):
            color = "white" if kl[l, t] > 0.6 * kl[1:].max() else "black"
            ax.text(t, l, model.tokenizer.decode(top[l, t]), ha="center", va="center", fontsize=8, color=color)
    ax.set_xticks(range(seq_len), model.to_str_tokens(text), rotation=45, ha="right")
    ax.set_yticks(range(n_rows), row_names)
    ax.set_xlabel("input position (each cell predicts the NEXT token)")
    fig.colorbar(im, ax=ax, label="KL from final prediction (embed row clipped)")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_logit_lens(model, "James Potter went to school. James")

In [ ]:
plot_logit_lens(model, "When Mary and John went to the store, John gave a drink to")

<details>
<summary><b>What to look for</b></summary>

- At which layer does " Potter" appear after the second " James"?
- When does " Mary" become the top prediction at the last position?
- What do the first few rows predict? Often just a copy of the current token or very frequent words.

Try your own prompts!
</details>

## Exploration: more than the top token

The top-1 token hides a lot. Here are the five most likely tokens, with probabilities, at a few depths.

In [ ]:
def show_top_k(model, text, k=5, layers=(0, 4, 8, 12)):
    log_probs = logit_lens(model, model.to_tokens(text))
    print(f"Next token after: {text!r}\n")
    for l in layers:
        values, indices = log_probs[l, -1].exp().topk(k)
        preds = ", ".join(f"{model.tokenizer.decode(i)!r} {v:.2f}" for v, i in zip(values, indices))
        print(f"{'embed' if l == 0 else f'after L{l - 1}':>9}: {preds}")

show_top_k(model, "The Eiffel Tower is located in the city of")

## Where the logit lens struggles

In the early and middle layers the logit lens often shows noise.
That is not necessarily because the model knows nothing yet.
The lens assumes that $h_\ell$ is already written in the "language" of the final layer, and simply skips everything the remaining layers would do with it.

Two ways to do better:
- the **tuned lens** learns a small linear correction for every layer,
- the **Jacobian lens** (next notebook) asks how the remaining layers actually *react* to $h_\ell$, on average, and uses that instead of skipping them.